In [1]:
from your_job_offer.services.vacancies_repository.db_methods import get_all_vacancies

vacancies = get_all_vacancies()

In [37]:
from your_job_offer.services.cv_parser.parser import ResumeParser

parser = ResumeParser()
user1 = parser.parse("your_job_offer/tests/parser/files/resume1.pdf")
user2 = parser.parse("your_job_offer/tests/parser/files/resume2.pdf")
user3 = parser.parse("your_job_offer/tests/parser/files/resume3.pdf")

Заметки:
- в requirments есть уровень образования
- если есть какой-то уровень образования, то есть и все ниже, чтобы предложения типа среднее образование и выше работали
- если мы возьмем слишком много вакансий, то ничего страшного, если упустим - пролема

Нужные поля:
- experience
- requirment
- area

In [11]:
from enum import Enum

from your_job_offer.domain.models.jobs import Vacancy
from your_job_offer.domain.models.user import User

In [ ]:
import re
from string import punctuation

from tqdm import tqdm

from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import download as nltk_download

from pymorphy2 import MorphAnalyzer

punctuation = punctuation.replace("+", "")


class TextPreprocessor:
    def __init__(self):
        nltk_download("punkt_tab")
        nltk_download("stopwords")
        rus_stops = stopwords.words("russian")
        self.filter_token = rus_stops + [
            "знание",
            "способность",
            "умение",
            "высокий",
            "уровень",
            "степень",
            "опыт",
            "хороший",
            "практический",
            "навык",
            "качество",
            "год",
            "мидло",
            "понимание",
            "концепция",
            "некоторый",
        ]
        self.parser = MorphAnalyzer()

    @staticmethod
    def clean(word: str) -> str:
        return re.sub(r"[^A-ZА-Яa-zа-я+\s]", "", word)

    def lemmatize(self, word: str) -> str:
        return self.parser.parse(word)[0].normal_form

    def text_to_tokens(self, text: str) -> list[str]:
        text = text.lower()
        text = text.translate(
            str.maketrans(punctuation, " " * len(punctuation))
        )
        text = text.translate(str.maketrans({"\n": " ", "\t": " ", "-": " "}))
        tokenized_text = word_tokenize(text)
        clean_text = list(map(self.clean, tokenized_text))
        lemmatized_text = []
        for word in clean_text:
            word = self.lemmatize(word)
            if not (
                len(word) < 2 or word in self.filter_token
            ):  # однобуквенные слова смысла не несут
                lemmatized_text.append(word)
        return lemmatized_text

In [ ]:
def isin(skills: list[str], requirement: str | None) -> bool:
    if requirement is None or len(requirement) == 0:
        return True
    for skill in skills:
        if skill in requirement:
            return True
    return False


def _match_vacancies_by_words_entry(
    vacancies: list[Vacancy], user: User
) -> list[Vacancy]:
    """
    [Baseline]
    проверяет, что хотя бы один скилл из user входит в хотя бы одно слово из requirement
    """
    processor = TextPreprocessor()
    user_skills = list(
        map(
            lambda skill: " ".join(processor.text_to_tokens(skill)),
            user.skills,
        )
    )
    print(user_skills)
    return list(
        filter(
            lambda vacancy: isin(user_skills, vacancy.requirement), vacancies
        )
    )

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import trange
from operator import itemgetter


def torch_set_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch_set_seed(42)
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny")


def embed_bert_cls(text):
    t = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        model_output = model(**{k: v.to(model.device) for k, v in t.items()})
    embeddings = model_output.last_hidden_state[:, 0, :]
    embeddings = torch.nn.functional.normalize(embeddings)
    return embeddings[0].cpu().numpy()


def cos_dist(x, y):
    return 1 - np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))


def _match_vacancies_by_bert(
    vacancies: list[Vacancy], user: User
) -> list[Vacancy]:
    """
    обрабатывает bertом, находит самые близкие вакансии(top100)
    """
    user_skill_embedding = embed_bert_cls(", ".join(user.skills))
    vacancies_skill_embeddings = np.zeros(
        (len(vacancies), len(user_skill_embedding))
    )
    for i in trange(len(vacancies)):
        vacancies_skill_embeddings[i] = embed_bert_cls(
            vacancies[i].requirement
        )
    dists = np.apply_along_axis(
        cos_dist, 1, vacancies_skill_embeddings, user_skill_embedding
    )
    indexes = np.argsort(dists)[:100]

    return itemgetter(*indexes)(vacancies)

In [ ]:
class MatchingEnum(Enum):
    WORDS_ENTRY = "words_entry"
    BERT = "bert"


def filter_without_skills(
    vacancies: list[Vacancy], user: User
) -> list[Vacancy]:
    """
    фильтрует то, что не смогли отфильтровать по запросам к бд, но без учёта скиллов,
    то есть поля area, experience
    """
    if user.relocation:
        return vacancies
    return vacancies  # TODO


def match_vacancies(
    vacancies: list[Vacancy],
    user: User,
    mode: MatchingEnum = MatchingEnum.WORDS_ENTRY,
) -> list[Vacancy]:
    """
    подбирает вакансии

    :param vacancies: отфильтрованные по полям employment, schedule, buisiness_trip_readiness, relocation, min_salary, max_salary вакансии
    :param mode: способ подбора
    :return: список отсортированных вакансий
    """
    vacancies = list(filter(lambda x: not x.requirement is None, vacancies))
    vacancies = filter_without_skills(vacancies, user)
    if mode == MatchingEnum.WORDS_ENTRY:
        return _match_vacancies_by_words_entry(vacancies, user)
    if mode == MatchingEnum.BERT:
        return _match_vacancies_by_bert(vacancies, user)


def print_result(func, user, print_count: int = 20):
    vacancies_ = func(vacancies, user)
    print(f"User skills: {user.skills}")
    print(f"Найдено {len(vacancies_)} из {len(vacancies)} вакансий")
    print(f"Примеры")
    for i in range(print_count):
        print(vacancies_[i].requirement)

In [ ]:
print_result(match_vacancies, user1)

['c++', 'python', 'go', 'git', 'docker', 'django', 'mpi', 'bash', 'virtualbox']
User skills: ['C/C++', 'Python', 'Go', 'Git', 'Docker', 'Django', 'MPI', 'Bash', 'VirtualBox']
Найдено 71 из 2500 вакансий
Примеры
Опыт на аналогичной позиции от полугода. Владение SQL, Python, google spreadsheets, навыки OSINT. Бизнес-ориентированный подход к выполняемым задачам.
Читает код на python или знает python. Имеет небольшой опыт работы с Apache Airflow. Был опыт работы с маркетплейсам. 
Владение SQL на продвинутом уровне. Опыт визуализации данных с помощью PowerBI или Tableau. Понимание продуктовых метрик в digital бизнесе. 
Высшее образование. Уверенное владение Python, SQL, bash. Опыт работы с BI системами (Datalens/Tableau/Redash). Понимание различий между метриками. 
Уверенные знания и опыт администрирования Linux. Опыт написания bash скриптов. Понимание принципов работы сетей. Уверенное знание и опыт в Kubernetes. 
Знание linux систем. Навыки автоматизации процесса разработки в docker. Знани

[nltk_data] Downloading package punkt_tab to /home/ruslan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ruslan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [70]:
print_result(lambda vacancies, user : match_vacancies(vacancies, user, MatchingEnum.BERT), user1)

100%|██████████| 2468/2468 [00:36<00:00, 67.36it/s] 


User skills: ['c++', 'Python', 'SQL', 'Docker', 'git', 'bash', 'многопоточные программы', 'алгоритмы и структуры данных']
Найдено 100 из 2500 вакансий
Примеры
Инструменты тестирования: Chrome dev tools, Postman, Fiddler, Swagger. Работа с данными, тестирование API и интеграций (REST API, SOAP, JSON, XML...
Знание теории и практики баз данных. Опыт построения реляционных баз данных, DWH. Знания Python, SQL, PostgreSQL, Airflow, Kafka, DBT, NiFi. 
Linux(Ubuntu/CentOS). Gitlab, Nexus, HELM. Unix Shell, Python, Ansible. Apache, Tomcat, Nginx, HAProxy. MySQL, PostgreSQL.
Мы используем: GKE, K8s, PHP, Symphony, PostgreSQL, Redis, Kafka, Google object storage, React, GitLab CI/CD, Ansible, Terraform...
1. Уверенное пользование Figma, Photoshop, illustrator, редакторами векторной графики. Владение PHP, MySQ L, HTML 4, CSS 2/3, javascript, jquery. 
Знание Linux — файловые системы, особенности работы с сетью, памятью, изоляция процессов, тюнинг, безопасность, анализ проблем: strace, tcpdump, per